In [ ]:
# [MARKDOWN CELL]
# # 03. Fan Array Aerodynamic Characterization & HWA Calibration
#
# **Purpose:** Experimental airflow calibration, axial decay curve fitting, duty cycle characterization, and error quantification for hot-wire anemometry (HWA).
#
# ---
# ## 1. PWM Duty Cycle Calibration (Air Velocity & Fan RPM)

# [CODE CELL]
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit

# Data
duty = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
velocity = np.array([0, 0.08, 0.35, 0.69, 0.94, 1.09, 1.26, 1.55, 2.11, 2.55, 2.71])
rpm  = np.array([0, 320, 420, 520, 600, 660, 780, 870, 1000, 1200, 1420])

# --- Polynomial Fit for Velocity ---
coeffs_vel = np.polyfit(duty, velocity, deg=2)
poly_vel = np.poly1d(coeffs_vel)
duty_smooth = np.linspace(0, 100, 300)

plt.figure(figsize=(8, 5))
plt.scatter(duty, velocity, s=70, label="Measured Velocity")
plt.plot(duty_smooth, poly_vel(duty_smooth), linewidth=2.5, label="2nd Order Poly Fit")
plt.xlabel("PWM Duty Cycle (%)", fontsize=12, fontweight='bold')
plt.ylabel("Velocity at 2 cm (m/s)", fontsize=12, fontweight='bold')
plt.title("Duty Cycle vs Air Velocity Characterization", fontsize=14, fontweight='bold')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Velocity Fit: V(D) = {coeffs_vel[0]:.4e} D² + {coeffs_vel[1]:.4e} D + {coeffs_vel[2]:.4e}")

# [MARKDOWN CELL]
# ## 2. Axial Velocity Decay Modeling
# Exponential decay of velocity along the fan axis: $V(x) = A e^{-kx}$

# [CODE CELL]
positions = np.array([2, 3, 4, 5, 6, 7, 8, 9])
forward = np.array([2.6, 2.1, 2.3, 1.7, 1.9, 1.1, 0.8, 0.9])
backward = np.array([2.4, 2.4, 1.8, 1.2, 1.0, 1.1, 0.9, 0.4])

def exp_decay(x, A, k):
    return A * np.exp(-k * x)

popt_f, _ = curve_fit(exp_decay, positions, forward, maxfev=5000)
popt_b, _ = curve_fit(exp_decay, positions, backward, maxfev=5000)

x_fit = np.linspace(2, 9, 300)

plt.figure(figsize=(8,6))
plt.scatter(positions, forward, s=60, label="Forward sweep data")
plt.scatter(positions, backward, s=60, label="Backward sweep data")

plt.plot(x_fit, exp_decay(x_fit, *popt_f), linewidth=2.5, label=f"Forward fit: A={popt_f[0]:.2f}, k={popt_f[1]:.2f}")
plt.plot(x_fit, exp_decay(x_fit, *popt_b), linewidth=2.5, linestyle='--', label=f"Backward fit: A={popt_b[0]:.2f}, k={popt_b[1]:.2f}")

plt.xlabel("Axial distance from fan exit (cm)", fontsize=12, fontweight='bold')
plt.ylabel("Velocity (m/s)", fontsize=12, fontweight='bold')
plt.title("Axial Velocity Decay Modeling", fontsize=14, fontweight='bold')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# [MARKDOWN CELL]
# ## 3. Hot-Wire Anemometer (HWA) Calibration & Error Analysis

# [CODE CELL]
data = {
    0.15: [0.16,0.16,0.18,0.18,0.12,0.05,0.15,0.15,0.15,0.16,0.18,0.2,0.18,0.16,0.07,0.02,0.05,0.11,0.12,0.13,0.1,0.16,0.17,0.15,0.11,0.14,0.12,0.16,0.16,0.17],
    0.42: [0.43,0.35,0.53,0.59,0.53,0.46,0.21,0.18,0.23,0.3,0.31,0.41,0.39,0.41,0.43,0.39,0.35,0.25,0.34,0.42,0.59,0.43,0.34,0.34,0.45,0.4,0.36,0.41,0.43,0.45,
           0.35,0.51,0.45,0.31,0.34,0.41,0.37,0.41,0.25,0.24,0.32,0.29,0.25,0.35,0.49,0.56,0.41],
    0.76: [0.89,0.97,1.02,0.96,0.96,0.92,0.89,0.71,0.71,0.76,0.72,0.52,0.52,0.65,0.43,0.43,0.75,0.75,0.64,0.64,0.64,0.78,0.59,0.59,0.76,0.63,0.63,0.6,0.55,0.62,
           0.62,0.46,0.4,0.72,0.72,0.78,0.74,0.65,0.65,0.55,0.54,0.6,0.68],
    1.00: [0.67,0.77,0.86,1.17,1.24,1.14,1.15,1.3,1.07,1.07,1.1,1.18,1.2,1.22,1.16,1.12,0.84,0.83,0.85,0.9,0.82,0.71,0.58,0.57,0.81,0.78,0.77,0.83,0.9,0.91,
           0.89,0.99,0.77,0.81,0.97,1.02,1.1,0.99,1.1,1.08,1,0.97,0.96,0.93,0.99,0.96,0.96,0.93,1.05,1.05,1.07,0.97,0.91,0.9,0.74,0.74,0.74,0.85,0.78,0.7,0.55],
    1.25: [1.21,1.11,1.21,1.15,1.28,1.22,1.2,1.04,1.21,1.17,1.05,1.27,1.29,1.29,1.27,1.05,1.07,1.05,1.08,1.17,1.01,1.08,1.27,1.31,1.17,1.15,1.32,1.31,1.29,1.36,
           1.39,1.4,1.3,1.28,1.26,1.38,1.43,1.43,1.44,1.25,1.23,1.23,1.32,1.41,1.22],
    1.55: [1.63,1.58,1.66,1.66,1.8,1.85,1.75,1.87,1.67,1.63,1.57,1.76,1.82,1.75,1.78,1.79,1.72,1.76,1.67,1.51,1.62,1.67,1.77,1.84,1.7,1.82,1.7,1.62,1.6,1.78,
           1.68,1.71,1.56,1.69,1.51,1.31,1.51,1.55,1.58,1.54,1.58,1.59,1.39,1.42,1.21,1.44,1.48,1.51,1.56,1.2,1.22,1.17,1.17],
    1.78: [1.88,1.74,1.76,1.71,1.67,1.67,1.62,1.67,1.7,1.73,1.74,1.75,1.81,1.88,1.91,1.82,1.8,1.73,1.73,1.75,1.81,1.77,1.79,1.81,1.78,1.96,1.89,1.84,1.8,1.77,
           1.76,1.77,1.69],
    2.49: [2.49,2.46,2.51,2.42,2.41,2.47,2.31,2.25,2.42,2.38,2.55,2.48,2.49,2.48,2.46,2.46,2.57,2.69,2.58,2.59,2.53,2.47,2.49,2.47,2.65,2.57,2.56,2.57,2.54,2.47,2.22,1.74,2.69]
}

vsets = sorted(data.keys())
mean_errors, std_errors, abs_errors = [], [], []

for vset in vsets:
    values = np.array(data[vset])
    errors = values - vset
    mean_errors.append(np.mean(errors))
    std_errors.append(np.std(errors))
    abs_errors.append(np.mean(np.abs(errors)))

# Bias Trend Visualization
plt.figure(figsize=(6,4))
plt.plot(vsets, mean_errors, marker='o', label="Mean Error")
coeffs_bias = np.polyfit(vsets, mean_errors, 2)
poly_bias = np.poly1d(coeffs_bias)
x_fit = np.linspace(min(vsets), max(vsets), 200)
plt.plot(x_fit, poly_bias(x_fit), linestyle='--', label='Quadratic Bias Trend')
plt.axhline(0, linestyle=':', color='black')
plt.title("Nonlinearity Check in Measurement Bias")
plt.xlabel("Vset [m/s]")
plt.ylabel("Mean Error [m/s]")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Mean Errors across set points:", mean_errors)
print("Quadratic Fit Coefficients:", coeffs_bias)